I have gathered the following notes from a video by Jim Blinn about his 
computer graphics system. This is the talk called "Magrathea" which is the
basis of this current project. Please reformat it into prose, appropriate 
for a (long) section of the introduction of my literate program 
documentation.
More grist for the mill. I watched Jim Blinn's 22-minute talk on Magrathea,
and have the following notes:
[Magrathea talk here](https://www.dropbox.com/scl/fi/zvmx5h03xjur6mq6iig5c/How-to-Make-a-Planet-Narration-Good-V3-640x480.mp4?rlkey=atatsxkaldf10req5dplp9v37&e=1&dl=0)

* Blinn was able to find most original data and source code, to remind 
  himself how things work.
* Talk called "Magrathea" because of 1979 talk with journalist from 
  England, who brought up Hitchhiker's guide as a BBC radio program.
* Blinn met Kohlhase shortly after being hired at JPL. Kohlhase and Paul
  Penzell already had a line drawing program used for visualizing what
  Voyager would see at Jupiter and Saturn. Most are lost.
* Seeing the Kohlhase animations inspired Blinn. His purpose in life became
  to bring those animations to life.
* Blinn hired by Bob Holzman. Holzman purchase an exact copy of the hardware
  Blinn used at UofU.
* Computer was PDP-11/55. 2MHz clock, 128KiB main memory, 16-bit address space.
  Half for OS, half for user code.
* Computer had two display peripherals: 
  * E&S Picture System 2 - vector drawing.
  * E&S Frame Buffer - color bitmap images. 
  Picture of desk shows dumb terminal with keyboard, E&S vector display with
  8 dials in front of it, color monitor for E&S Frame Buffer.
    

* Frame buffer hardware: 256KiB memory, 2048x 1Kib chips for memory. 8KiB 
  at a time accessable to PDP-11 through a memory mapping into user address 
  space. Output was a microprocessor in the frame buffer hardware, 
  continually reading frame buffer memory a pixel at a time. Pixel is 8-bit
  index into 256-entry palette. Each palette entry is 16 bit, 5 bit red, 
  6 green, 5 blue.
* Palette for Jupiter is 32 shades (lightness) of 8 saturations of orange 
  (hue).
* Blinn used ordered dither to improve color resolution at expense of spatial
  resolution
* Blinn had issues with scaling. In a single image, there is a meter-scale
  spacecraft 10 m away from camera, a moon of thousands of kilometer radius
  hundreds of thousands of km away, and a planet of tens of thousands of
  kilometer radius millions of kilometers away. Use depth priority and
  scaling tailored for each layer
* Frame buffer synergy -- OS doesn't manipulate nor reset frame buffer. A 
  user program can run to completion and draw on the frame buffer, and when
  it finishes, its output remains in the frame buffer. The next program can
  then *edit* or *draw over* the frame buffer with its own work. Frame buffer
  synergy is then "A collection of programs that all 'do something' to the 
  Frame Buffer".
    * STARDRW - draw star background
    * PLANET - draw texture mapped ellipsoides
    * POLYS - draw polygonal meshes
    * MSC - draw other stuff like sunburst etc
    * RINGS - draw rings
    * SPACE - orchestrate. Also works with vector graphics display to allow
      user to tune camera parameters.

* All programs had a common command language and common interpreter. 
  Commands can come from keyboard, or `READ` (include) a file, which can
  recursively include more files. Library routines for CLI (command language
  interpreter), nested transforms, lighting, math, primitive rendering
  objects (lines, dots).
* Orbital elements - from mission planning team. Part of library accepted
  elliptical or hyperbolic orbital elements and time, and returned position.
  [`spkezr()` and spice kernels are modern replacement.]
* Spacecraft needs orientation simulation. HGA axis pointed at Earth. Star
  tracker towards Canopus. [This constrains the spacecraft position like 
  Point Toward.]
* Spacecraft scan platform and its azimuth and elevation actuators.
* Blinn independently implemented spacecraft and scan platform pointing
  algorithm. Can track what mission design is actually doing, or can
  reverse -- Camera is desired to point at moon at given time -- figure out
  pointing angles.
* Code can simulate narrow FOV of cameras, "looking down" their telescopes,
  or plotting their fields of view. One mode of SPACE allowed manual control
  of simulated az and EL and real-time plot of footprint.

* Sample of viewing specification (with #comments by me):

```
TIME 0 # from closest approach
FOV 20 # deg
FROM VOYAGER
DIST .2,.8,0
AT JUPITER,.4,-.05

AT #,x,y,z

FROM VOYAGER
FIXF x,y,z

SIT 180., 100.137,
    181.497, 89.0488,
    1500,
```

Display has picture of Voyager with Jupiter in background and HUD showing
`TIME: 0 DAYS 0: 0: 9` and:
```
KNOB 5: Dst For #Distance to foreground object
KNOB 4: Y Main # Y position of main (background planet) object
KNOB 3: X Main # X position of background object
KNOB 2: Y Foreg # Y position of foreground (spacecraft) object
KNOB 1: X Fore  # X position of foreground object
KNOB 0: FOV
```

SPACE basically works exactly like I guessed: Manually twist knobs to place
background planet or moon at chosen spot. This constrains the camera. Then
manually twist other knobs to position the foreground spacecraft. Also a
topocentric mode for surface of moon.

* SPACE knew about where each object is. It can transform each object to
  viewing space, sort the objects by depth, do FOV culling, then traverse
  the list from back to front. SPACE wrote command files for the system.
  An example is:

``` DRAW.CMD
MSC CLR 0 # use MSC to clear the frame buffer by writing 0 to all pixels. This is a single command on MSC command line.
STARDRW READ STARS.TMP # Use the stardraw command. Its command is a recursive read of STARS.TMP command file.
PLANET READ JUPIT.TMP
MSC DOT x,y,brt # for moons less than 1 pixel in size, instead of PLANET
PLANET READ IO.TMP
POLYS READ VOYAG.TMP
MAN SAVE file.PIC # Invoke a command that *reads* the frame buffer and writes it to a file.
```

``` STARS.TMP
MAT #,#,#,#...
READ STARS.DAT # STARS.DAT is the star catalog but reformatted as commands itself.
```

```JUPIT
MAT #,#,#,#...
... # Things like load the texture map
DRAW
```

```IO
MAT #,#,#,#...
...
DRAW
```

```VOYAG
MAT #,#,#,#...
...
READ VOYAGER.PLY # Polygon list in the form of commands
```

Since the frame buffer was connected to a live display, it would continuously
update as programs acted on it. The example shows stars, planet, moon, 
spacecraft appearing in order.
* STARDRW used effectively a CSV with the following columns:
   * Magnitude
   * String containing encoded Bayer name and spectral type
   * Three elements of unit direction vector to star
* 422 brigthest stars used. Drawn as 3x3 point spread function. Had to 
  tinker with magnitude-brightness relation as reality would cause dimmer
  stars to be invisible.
* MSC could do dots of arbitrary size. It shows a sunburst as a horizontal
  set of rays, vertical set of rays, and diagonal rays that are much shorter.
  Later sunburst used more complicated effects.
* PLANET used a tradeoff. Instead of chopping sphere up into a trianglular
  mesh ("a zillion little triangles"), write one more complicated program to
  handle one more complicated primitive, the texture-mapped ellipsoid.
* Primitive is one sphere of unit radius at origin, which can be arbitrarily
  transformed.
* First step -- figure out boundary of ellipsoid on screen, so pixels
  outside of ellipsoid don't need to be considered. [This is an example
  where I just use brute force -- evaluate the quadratic at every pixel
  and just ignore pixels with no real quadratic roots.]
* Second step -- iterate scan lines. For each line, figure out x extent then
  iterate those pixels. Figure out quadratic equation for each pixel actually
  on the ellipsoid, and evaluate where on the ellipsoid we hit in order to
  index the texture map, look up the normal, etc.
* Optimized depth calculation. For Narrow Angle Camera (NAC) simulation, 
  quadratic equation involved two nearly equal roots very far away, required
  care.
* Edge case (literally): Sometimes the planet was on the edge of the screen,
  and its projection onto the screen plane isn't an ellipse but a hyperbola.
  Hyperbola math turned into its own primitive to draw bow shock on Saturn
  approach video.
* Anti-aliasing the edge. For pixels on the edge, approximate how much of the
  pixel is covered by the disk, then do an alpha blend between what was
  previously at that pixel and what color the ellipsoid will be. [My solution
  is yet more brute force -- draw the image at 4x scale and then downsample it
  after all pipeline phases have finished.]
* Texture maps. For Voyager 1 movie, use approach observatory phase images
  taken by Voyager itself. Use the camera model and pointing to do a reverse
  texture map -- figure out where on the texture map each real image pixel
  maps to, then paint the texture map that color. Ended up using a composite
  of 4 images to cover the whole planet, then a paint program to manually
  fill in the gaps and smooth the seams. Bjorn Jonsson has done the modern
  form of this to generate the texture map I am using from closer approach
  images again from Voyager 1.
* Moon texture maps. Before Voyager 1, only rough color was known from ground
  and Pioneer observations. Rick Sternbach hired to imagine and paint
  surfaces for these moons. When real images came down, same reverse texture
  map technique used to make updated maps.
* POLYS - Blinn used a blueprint of the spacecraft. [It looks suspiciously
  exactly like the one I have. I think there is *one* blueprint that is so
  useful that everyone uses it for model construction.] He had a digitizer
  pen, but mostly used the labeled dimensions and typed numbers into his 
  database. [Similarly I use Inkscape to digitize points that don't have
  dimensions, but since my blueprint image has nonlinear distortions, I
  use dimensions when possible as well. I type my stuff into an scad model,
  and use OpenSCAD to make a perfect triangle mesh out of it.] He used his
  database to export triangles, tesselate cylinders into triangles, use
  special "tube" primitives to handle the structural tubes, and a shape
  of revolution primitive to handle the high-gain antenna.
* POLYS - database includes PNT lines for vertices and POLY lines describing
  closed polys (not always triangles).
* POLY rendering. 
  * Pass 1 - read, transform, light, cull, and put in a list. 
  * Pass 2 - sort list on max Y coordinate. 
  * Pass 3 - For each line in the frame buffer covered by a polygon, render
    the polygon(s) on that line.
* Memory limit -- only about 100 polygons fit into user memory. Had separate
  front side/back side models.
* POLYS eventually split into 3 programs, one for each pass -- PY1, PY2, PY3.
  This shrank the code, expanding data space such that up to 1500 polygons
  could be included simultaneously.
* Approach to Saturn, with scan platform motion and plasma scan roll. These
  maneuvers came from Kohlhase at mission planning. [We use ck files for
  spacecraft and independently for scan platform.]
* Polygonal rendering of Mimas. Texture mapping didn't work well as
  topography departed too much from ellipsoidal. Make a texture map, but
  also a manual displacement map. Do the tesselation of the sphere into
  triangles, then modify the radius of the vertices with the displacement
  map. TERRAIN program run in place of PY1 and PY2, result of TERRAIN piped
  to PY3.
* FBDRAW - anti-aliased line drawing. Used to draw field of view footprints.
  [**Therefore, this is the program that _got_ me when he used it for the 
  Miranda flyby.**] Also used to draw volcano plumes on Io, and visual 
  streaks for crossing ring plane.
* RINGS - draw rings. Do ray/plane solution, then get distance from center
  to intersection. Use this in a 1D table of transparency, and do an alpha
  blend of the previous frame buffer pixel and pure white. In order to do
  the painter's algorithm, invoke the program twice -- once drawing only 
  points that are on the far side of the planet, then call PLANET to draw
  the planet over the back of the rings, then draw the front side over the
  planet. Ring shading also uses planet eclipse code to figure the shadows
  of the planet on the rings, and can draw shadows on an ellipsoid (shading
  the existing pixels) to draw the shadows of the rings on the planet.
  Initial transparency map had 83 points. Studied papers [including some
  by Esposito at LASP] to come up with a brightness vs tau for both lit and
  unlit sides of rings. Lit side increases asymtotically as tau increases
  without limit, while unlit side increases to a maximum and then decreases
  asymtotically to zero as tau increases without limit.
* Moon shadows. First pass - use line from Sun through Moon. If ray
  intersects planet on lit side, use MSC to draw a black dot after PLANET.
* Ring shadows -- work with projected ellipses. Project cone with vertex at
  Sun and tangent to planet. Intersect this with the ring plane to form an
  ellipse. Project to screen to get another ellipse. Points inside this
  ellipse are in shadow and get dimmed.
* Moon eclipse experiment. Each point on moon is used as a viewpoint. Look
  at fraction of sun covered by planet and shade spot accordingly. [This is
  actually implemented in my project already.]
* Similar for Jupiter and Uranus rings. For Uranus ring occultation of sun,
  manually determine which frames have sun obscured by rings and dim the 
  sun at those points.
* 1983 - using a VAX with 2MB memory at this point. Able to accomodate a 
  ring model from Voyager 2 data with 20,000 transparency points.